<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/Hancock_GPT2LoRA_Narrative_Generative.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#GPT2LoRA Generative for Hancock (last 4 layers)

Install torchao To avoid this error: ImportError: Found an incompatible version of torchao

In [ ]:
%pip -q install -U "torchao>=0.16.0"

In [ ]:
# -*- coding: utf-8 -*-
# HANCOCK V3.2: Generative GPT-2 LoRA (last 4 blocks), answer-only training
# 1. Imports, reproducibility and configuration
import os, random, json, inspect, gdown
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from peft import LoraConfig, TaskType, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

SEED = 42
BATCH_SIZE = 64
EPOCHS = 250
PATIENCE = 30
LORA_LR = 1e-4
LORA_R = 8
LORA_ALPHA = 16
LORA_LAYERS = [8, 9, 10, 11]
WEIGHT_DECAY = 1e-4
MODEL_NAME = "openai-community/gpt2"

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 2. LOAD AND MERGE HANCOCK CLINICAL/PATHOLOGICAL DATA
# ============================================================
CLINICAL_FILE = "clinical_data.json"
PATHOLOGICAL_FILE = "pathological_data.json"

if not os.path.exists(CLINICAL_FILE):
    if gdown.download(id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q", output=CLINICAL_FILE, quiet=False) is None:
        raise FileNotFoundError("Could not download clinical_data.json.")

if not os.path.exists(PATHOLOGICAL_FILE):
    if gdown.download(id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F", output=PATHOLOGICAL_FILE, quiet=False) is None:
        raise FileNotFoundError("Could not download pathological_data.json.")

def load_json_table(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return pd.DataFrame(obj)
    if isinstance(obj, dict):
        for key in ["data", "records", "patients", "items"]:
            if key in obj and isinstance(obj[key], list):
                return pd.DataFrame(obj[key])
        try:
            return pd.DataFrame(obj)
        except ValueError:
            pass
    raise ValueError(f"Could not convert {path} to a table.")

clinical_df = load_json_table(CLINICAL_FILE)
pathological_df = load_json_table(PATHOLOGICAL_FILE)
if ("patient_id" not in clinical_df.columns or "patient_id" not in pathological_df.columns):
    raise KeyError("Both JSON files must contain 'patient_id'.")

df = clinical_df.merge(pathological_df, on="patient_id", how="left", suffixes=("", "_path"))
print("Clinical shape:", clinical_df.shape)
print("Pathology shape:", pathological_df.shape)
print("Merged shape:", df.shape)
# ============================================================
# 3. SELECT SIX FEATURES AND DEFINE THE HPV TARGET
# ============================================================
def resolve_col(candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Required column not found: {candidates}. " f"Available columns: {df.columns.tolist()}")
    return None

target_col = resolve_col(["hpv_association_p16", "hpv_association_p16_path"], required=True)
feature_candidates = {
    "age_at_initial_diagnosis": ["age_at_initial_diagnosis", "age_at_initial_diagnosis_path"],
    "sex": ["sex", "sex_path"],
    "smoking_status": ["smoking_status", "smoking_status_path"],
    "primary_tumor_site": ["primary_tumor_site", "primary_tumor_site_path"],
    "primarily_metastasis": ["primarily_metastasis", "primarily_metastasis_path"],
    "pN_stage": ["pN_stage", "pN_stage_path", "pathological_N_stage", "pathological_n_stage"]
}
feature_map = {name: resolve_col(cols, required=True) for name, cols in feature_candidates.items()}
features = list(feature_candidates.keys())
df = df.dropna(subset=[target_col]).copy()

def normalize_target(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["negative", "neg", "0", "false", "no"]:
        return 0
    if s in ["positive", "pos", "1", "true", "yes"]:
        return 1
    try:
        n = float(s)
        if n in [0, 1]:
            return int(n)
    except ValueError:
        pass
    return np.nan

df["HPV_Label"] = df[target_col].apply(normalize_target)
df = df.dropna(subset=["HPV_Label"]).copy()
df["HPV_Label"] = df["HPV_Label"].astype(int)
feature_df = df[[feature_map[c] for c in features]].copy()
feature_df = feature_df.rename(columns={feature_map[c]: c for c in features})
df = pd.concat([feature_df, df[["patient_id", "HPV_Label"]]], axis=1)
# A row-level split is valid only if there is one eligible row per patient.
if df["patient_id"].isna().any() or df["patient_id"].duplicated().any():
    raise ValueError("Missing or duplicate patient IDs; resolve before patient-level splitting.")
df = df.loc[:, ~df.columns.duplicated()].copy()
df["age_at_initial_diagnosis"] = pd.to_numeric(df["age_at_initial_diagnosis"], errors="coerce")
print("Patients:", len(df))
print("HPV classes (0=negative, 1=positive):")
print(df["HPV_Label"].value_counts().sort_index())
print("Features:", features)
print("N-stage source:", feature_map["pN_stage"])
# ============================================================
# 4. STRATIFIED TRAIN/VALIDATION/TEST SPLIT (65/15/20)
# ============================================================
train_val_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["HPV_Label"])
train_df, val_df = train_test_split(train_val_df, test_size=0.1875, random_state=SEED, stratify=train_val_df["HPV_Label"])
train_df, val_df, test_df = (split.copy() for split in [train_df, val_df, test_df])
# ============================================================
# 5. IMPUTE MISSING VALUES USING TRAINING DATA ONLY
# ============================================================
fill_values = {}
for c in features:
    if c == "age_at_initial_diagnosis":
        value = train_df[c].median()
        if pd.isna(value):
            value = 0
    else:
        # Preserve original category values, including numeric-coded categories.
        modes = train_df[c].dropna().mode()
        value = modes.iloc[0] if not modes.empty else "Unknown"
    fill_values[c] = value
    for split in [train_df, val_df, test_df]:
        split[c] = split[c].fillna(value)
        if split[c].dtype == object:
            split[c] = split[c].astype(str).replace({"": "Unknown", "nan": "Unknown", "None": "Unknown"})

y_train = train_df["HPV_Label"].to_numpy(dtype=int)
y_val = val_df["HPV_Label"].to_numpy(dtype=int)
y_test = test_df["HPV_Label"].to_numpy(dtype=int)
print("Train:", len(train_df), np.bincount(y_train, minlength=2))
print("Validation:", len(val_df), np.bincount(y_val, minlength=2))
print("Test:", len(test_df), np.bincount(y_test, minlength=2))

# Shared category vocabulary: unsupported values are unknown in every representation.
category_mapping = {
    "sex": {"female": 0, "male": 1},
    "smoking_status": {"non-smoker": 0, "former": 1, "smoker": 2},
    "primary_tumor_site": {"CUP": 0, "Oropharynx": 1},
    "primarily_metastasis": {"no": 0, "yes": 1},
    "pN_stage": {"pN0": 0, "pN1": 1, "pN2": 2, "pN2a": 3, "pN2b": 4,
                 "pN2c": 5, "pN3": 6, "pN3b": 7, "NX": 8}
}
for frame in (train_df, val_df, test_df):
    for col, mapping in category_mapping.items():
        frame[col] = frame[col].where(frame[col].isin(mapping), "Unknown")

# Initialise generative GPT-2 with trainable LoRA adapters; pretrained weights remain frozen.
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
model.requires_grad_(False)
lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
                         target_modules=["c_attn"], layers_to_transform=LORA_LAYERS, layers_pattern="h",
                         lora_dropout=0.0, bias="none", fan_in_fan_out=True)
model = get_peft_model(model, lora_config)
model.eval()  # Keep backbone dropout disabled, as in the previous LoRA experiment.
assert all(p.requires_grad == ("lora_" in name) for name, p in model.named_parameters())
trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
assert trainable_names and all(any(f".h.{i}." in name for i in LORA_LAYERS) for name in trainable_names)
assert all(any(f".h.{i}." in name for name in trainable_names) for i in LORA_LAYERS)
model.print_trainable_parameters()

# 6. Clinical narrative representation
MODEL_LABEL = f"GPT2_Narrative_V3_2_Generative_LoRA_Last4_lr{LORA_LR:g}_r{LORA_R}"

smoking_description = {
    "non-smoker": "The patient is a non-smoker, with no reported smoking history and lower tobacco-related cancer risk.",
    "former": "The patient is a former tobacco smoker, with past exposure and residual tobacco-related cancer risk.",
    "smoker": "The patient is a current tobacco smoker, with ongoing exposure and elevated tobacco-related cancer risk."
}

site_description = {
    "CUP": "The patient has carcinoma of unknown primary (CUP), with no identified primary tumour site.",
    "Oropharynx": "The primary tumour originates in the oropharynx."
}

metastasis_description = {
    "no": "No metastases were recorded at initial diagnosis.",
    "yes": "Metastases were present at initial diagnosis."
}

def patient_to_text(row):
    age = row["age_at_initial_diagnosis"]
    sex = row["sex"]
    smoking = row["smoking_status"]
    site = row["primary_tumor_site"]
    metastasis = row["primarily_metastasis"]
    stage = row["pN_stage"]

    if stage == "pN0":
        nodal = "No regional lymph node metastases were identified (pN0)."
    elif stage == "NX":
        nodal = "Regional lymph node involvement could not be assessed (NX)."
    elif stage in {"pN1", "pN2", "pN2a", "pN2b", "pN2c", "pN3", "pN3b"}:
        nodal = f"Regional lymph node metastases are present (pathological stage {stage})."
    else:
        nodal = f"The pathological regional lymph node stage is {stage}."

    # narrative = (
    #     f"The patient is a {age}-year-old {sex} diagnosed with head and neck cancer. "
    #     f"{site_description.get(site, 'Primary tumour site is unknown.')} "
    #     f"{metastasis_description.get(metastasis, 'Metastasis status is unknown.')} "
    #     f"{nodal} "
    #     f"{smoking_description.get(smoking, 'Smoking status is unknown.')}"
    # )
    narrative = (
        f"The patient is a {age}-year-old {sex} diagnosed with head and neck cancer. "
        f"{site_description[site]} "
        f"{metastasis_description[metastasis]} "
        f"{nodal} "
        f"{smoking_description[smoking]}"
    )

    question = (
        "\nQuestion: Based on the clinical findings, predict whether "
        "the tumour is HPV positive or HPV negative. "
        "Answer with only one word: positive or negative.\nAnswer:"
    )
    return narrative + question

X_train, X_val, X_test = [df.apply(patient_to_text, axis=1).tolist() for df in (train_df, val_df, test_df)]
print("Example text:", X_train[0])

# 7. Tokenise prompts and identify the one-token answer for each class

tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Align the final prompt token across the batch.
model.config.pad_token_id = tokenizer.pad_token_id
answer_words = [" negative", " positive"]  # Class 0 and class 1; leading space is important.
answer_ids = [tokenizer.encode(word, add_special_tokens=False) for word in answer_words]
assert all(len(ids) == 1 for ids in answer_ids), "Each answer must be exactly one GPT-2 token."
assert answer_ids[0] != answer_ids[1], "The answer tokens must be different."
answer_token_ids = torch.tensor([ids[0] for ids in answer_ids], dtype=torch.long, device=device)
print("Answer tokens:", dict(zip(answer_words, answer_token_ids.tolist())))

def tokenize_texts(texts):
    encoded = tokenizer(texts, padding=False, add_special_tokens=False)["input_ids"]
    for text, ids in zip(texts, encoded):
        assert len(ids) < model.config.n_positions, "Prompt leaves no room for the answer token."
        # Verify that the target really is the next token after the complete prompt.
        for word, target in zip(answer_words, answer_ids):
            assert tokenizer.encode(text + word, add_special_tokens=False) == ids + target
    return encoded

# 8. Create DataLoaders with dynamic left padding

def collate_batch(batch):
    ids, labels = zip(*batch)
    encoded = tokenizer.pad({"input_ids": list(ids)}, padding=True, return_attention_mask=True, return_tensors="pt")
    return encoded["input_ids"], encoded["attention_mask"], torch.tensor(labels, dtype=torch.long)

def make_loader(X, y, shuffle=False):
    dataset = list(zip(tokenize_texts(X), y.tolist()))
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(SEED))

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)

# 9. Answer-only causal language modelling loss and optimiser (no MLP classifier)

class_counts = np.bincount(y_train, minlength=2)
assert np.all(class_counts > 0), "Both classes must be present in training data."
class_weights = torch.tensor(len(y_train) / (2 * class_counts), dtype=torch.float32, device=device)
print("Class weights:", class_weights)

# Newer Transformers versions can compute only the final-token vocabulary logits.
# Older versions fall back to the standard forward pass.
supports_logits_to_keep = "logits_to_keep" in inspect.signature(model.get_base_model().forward).parameters

def forward(input_ids, attention_mask):
    position_ids = (attention_mask.cumsum(dim=1) - 1).masked_fill(attention_mask == 0, 0)
    kwargs = {"logits_to_keep": 1} if supports_logits_to_keep else {}
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, position_ids=position_ids,
                    use_cache=False, **kwargs)
    return outputs.logits[:, -1, :]  # Shape: (batch, vocabulary size)

def criterion(logits, labels):
    # Predict the single answer token across the FULL vocabulary, not just two classes.
    # Equivalent to shifted causal LM loss with all prompt/padding targets masked (-100).
    loss = nn.functional.cross_entropy(logits, answer_token_ids[labels], reduction="none")
    return (loss * class_weights[labels]).mean()

lora_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(lora_params, lr=LORA_LR, weight_decay=WEIGHT_DECAY)
print("LoRA trainable parameters:", sum(p.numel() for p in lora_params))
print("Total trainable parameters:", sum(p.numel() for p in lora_params))

# 10. Train and select best model

def validate(model, dataloader, criterion):
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X, mask, y in dataloader:
            X, mask, y = X.to(device), mask.to(device), y.to(device)
            val_loss += criterion(forward(X, mask), y).item() * len(y)
    return val_loss / len(dataloader.dataset)

best_val_loss = float("inf")
best_weights = None
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    model.eval()  # Same dropout policy as previous LoRA; gradients are enabled below.
    train_loss = 0
    for X, mask, y in train_loader:
        X, mask, y = X.to(device), mask.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(forward(X, mask), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(y)
    train_loss /= len(train_loader.dataset)
    val_loss = validate(model, val_loader, criterion)

    if np.isfinite(val_loss) and val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = {k: v.detach().cpu().clone() for k, v in get_peft_model_state_dict(model).items()}
        best_epoch = epoch
    if epoch % 20 == 0:
        print(f"Epoch {epoch}/{EPOCHS}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Best Epoch={best_epoch}")
    if epoch - best_epoch >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (no validation loss improvement for {PATIENCE} epochs).")
        break

if best_weights is None:
    raise RuntimeError("No finite best validation loss was obtained.")
set_peft_model_state_dict(model, best_weights)
model.save_pretrained(f"{MODEL_LABEL}_best_adapter")
tokenizer.save_pretrained(f"{MODEL_LABEL}_best_adapter")
print(f"Best epoch: {best_epoch}, Best validation loss: {best_val_loss:.4f}")

# 11. Test evaluation: constrained one-token generation

model.eval()
predictions, probabilities, labels, generated_answers = [], [], [], []
allowed_ids = answer_token_ids.tolist()
with torch.no_grad():
    for X, mask, y in test_loader:
        X, mask = X.to(device), mask.to(device)
        # Greedy generation is restricted to the two valid answer tokens.
        outputs = model.generate(input_ids=X, attention_mask=mask, max_new_tokens=1, do_sample=False,
                                 prefix_allowed_tokens_fn=lambda batch_id, ids: allowed_ids,
                                 pad_token_id=tokenizer.pad_token_id, return_dict_in_generate=True,
                                 output_scores=True)
        new_tokens = outputs.sequences[:, X.size(1):]
        token_ids = new_tokens[:, 0]
        assert torch.isin(token_ids, answer_token_ids).all(), "Unexpected generated answer."
        predictions.extend((token_ids == answer_token_ids[1]).long().cpu().numpy())
        # Conditional P(positive | positive or negative) for ROC-AUC.
        scores = outputs.scores[0][:, answer_token_ids].float()
        probabilities.extend(scores.softmax(dim=1)[:, 1].cpu().numpy())
        labels.extend(y.numpy())
        generated_answers.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))

acc = accuracy_score(labels, predictions)
bal_acc = balanced_accuracy_score(labels, predictions)
auc = roc_auc_score(labels, probabilities)

print("Example generated answers:", generated_answers[:5])
print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"ROC-AUC:           {auc:.4f}")

Device: cuda
Clinical shape: (763, 32)
Pathology shape: (763, 18)
Merged shape: (763, 49)
Patients: 332
HPV classes (0=negative, 1=positive):
HPV_Label
0    191
1    141
Name: count, dtype: int64
Features: ['age_at_initial_diagnosis', 'sex', 'smoking_status', 'primary_tumor_site', 'primarily_metastasis', 'pN_stage']
N-stage source: pN_stage
Train: 215 [123  92]
Validation: 50 [29 21]
Test: 67 [39 28]


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 98,304 || all params: 124,538,112 || trainable%: 0.0789
Example text: The patient is a 53-year-old male diagnosed with head and neck cancer. The primary tumour originates in the oropharynx. No metastases were recorded at initial diagnosis. No regional lymph node metastases were identified (pN0). The patient is a current tobacco smoker, with ongoing exposure and elevated tobacco-related cancer risk.
Question: Based on the clinical findings, predict whether the tumour is HPV positive or HPV negative. Answer with only one word: positive or negative.
Answer:
Answer tokens: {' negative': 4633, ' positive': 3967}
Class weights: tensor([0.8740, 1.1685], device='cuda:0')
LoRA trainable parameters: 98304
Total trainable parameters: 98304
Epoch 20/250: Train Loss=0.7450, Val Loss=0.7343, Best Epoch=20
Epoch 40/250: Train Loss=0.7046, Val Loss=0.7020, Best Epoch=40
Epoch 60/250: Train Loss=0.6980, Val Loss=0.6964, Best Epoch=58
Epoch 80/250: Train Loss=0.6907, Val Loss=0.6908, B